In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gammaln

In [2]:
# Read the data
data = pd.read_csv("bird_count.csv")

year = data["yr"].values
count = data["count"].values

In [3]:
# Center the year variable for numerical stability
year_c = year - np.mean(year)


# Define the negative log-likelihood
# Y_i ~ Poisson(lambda_i)
# log(lambda_i) = beta0 + beta1 * year_c

def neg_log_likelihood(beta, year, count):
    
    beta0, beta1 = beta
    
    # Poisson mean
    lam = np.exp(beta0 + beta1 * year)
    
    # Log-likelihood
    log_likelihood = np.sum(
        count * np.log(lam)
        - lam
        - gammaln(count + 1)
    )
    
    # Minimize negative log-likelihood
    return -log_likelihood

In [4]:
# Estimate the parameters using maximum likelihood

initial_beta = np.array([1.0, 0.0])

result = minimize(
    neg_log_likelihood,
    initial_beta,
    args=(year_c, count)
)


# Check optimization
print("Optimization successful:", result.success)


# Estimated parameters
beta0, beta1 = result.x

print("Estimated parameters:")
print("beta0 =", beta0)
print("beta1 =", beta1)

Optimization successful: True
Estimated parameters:
beta0 = 2.1082121961214266
beta1 = -0.03244316557337736


In [5]:
# Calculate fitted Poisson means

lambda_hat = np.exp(beta0 + beta1 * year_c)

print("\nFitted mean counts:")
print(lambda_hat)


# Generate three hypothetical samples

np.random.seed(123)

sample_1 = np.random.poisson(lambda_hat)
sample_2 = np.random.poisson(lambda_hat)
sample_3 = np.random.poisson(lambda_hat)


# Combine the three samples

samples = pd.DataFrame({
    "sample": np.repeat([1, 2, 3], len(year)),
    "yr": np.tile(year, 3),
    "count": np.concatenate([
        sample_1,
        sample_2,
        sample_3
    ])
})


print("\nGenerated samples:")
print(samples)


# Save the samples as a CSV file

samples.to_csv("poisson_samples.csv", index=False)

print("\nSamples saved as poisson_samples.csv")


Fitted mean counts:
[ 6.9310676   7.15962082  9.28130671  8.151726    7.63958554  6.70981037
  9.90350443  8.42053065  7.89150227  8.69819919 10.2300743   9.58735949
  7.39571063]

Generated samples:
    sample    yr  count
0        1  2011      9
1        1  2010      6
2        1  2002     11
3        1  2006      8
4        1  2008      8
5        1  2012      8
6        1  2000     13
7        1  2005      6
8        1  2007      9
9        1  2004      9
10       1  1999      9
11       1  2001     10
12       1  2009      8
13       2  2011     11
14       2  2010      6
15       2  2002      9
16       2  2006     10
17       2  2008     10
18       2  2012      6
19       2  2000      9
20       2  2005      8
21       2  2007      5
22       2  2004     10
23       2  1999     13
24       2  2001     10
25       2  2009      5
26       3  2011      7
27       3  2010      4
28       3  2002     13
29       3  2006      6
30       3  2008      5
31       3  2012      2
32     